# FreshRetailNet-50K: dự báo nhu cầu 7 ngày có xét hết hàng và quyết định tồn kho

Notebook này chạy thiết kế ở mục 6–7 của báo cáo đề xuất "Dự báo nhu cầu hàng tươi sống có xét hiện tượng hết hàng phục vụ quản lý tồn kho: so sánh LightGBM và LSTM trên FreshRetailNet-50K".

| Thí nghiệm | Mô hình | Mục tiêu huấn luyện | Đặc trưng |
|---|---|---|---|
| E1 | Seasonal Naive (theo tuần) | doanh số thô | Y(T+h−7) |
| E2 | LightGBM | doanh số thô | nhu cầu quá khứ, lịch, khuyến mãi, thời tiết |
| E3 | LightGBM | nhu cầu khôi phục | như E2, thêm đặc trưng hết hàng trễ |
| E3m | LightGBM | doanh số thô, trọng số 0 cho ngày có hơn k giờ hết hàng | như E3 |
| E4 | LSTM, cửa sổ 28 ngày | doanh số thô | như E2 |
| E5 | LSTM, cửa sổ 28 ngày | nhu cầu khôi phục | như E3 |
| A1 | LightGBM | nhu cầu khôi phục | như E3, bỏ thời tiết |
| A2 | LightGBM | nhu cầu khôi phục | như E3, bỏ discount và activity_flag |

Mọi con số đều sinh ra khi chạy. Không có kết quả nào được ghi sẵn.

Chạy lần lượt từ trên xuống. Muốn đổi quy mô, sửa ô CONFIG hoặc truyền tham số qua papermill.

In [ ]:
# Colab: bỏ comment dòng dưới (Colab đã có numpy, pandas, pyarrow, torch bản CUDA, huggingface_hub; KHÔNG cài lại torch)
# !pip install -q lightgbm==4.6.0 psutil
# Máy cục bộ: pip install -r requirements.txt (xem README_RUN.md)

## 0. Cấu hình

Các tham số chính nằm ở ô dưới. Papermill có thể ghi đè các tham số này.

In [ ]:
# ===== CONFIG (papermill parameters) =====
SUBSET_N_SERIES = 5000        # None: toàn bộ 50,000 chuỗi; báo cáo đề xuất tập con ~5,000 chuỗi phân tầng
DEVICE = "auto"               # "auto" -> cuda / mps / cpu
LSTM_EPOCHS = 15              # số epoch tối đa (early stopping trên validation)
LSTM_PATIENCE = 3
LSTM_BATCH_SIZE = 1024
LSTM_HIDDEN = 64
LSTM_LAYERS = 1               # báo cáo: 1–2 lớp
LSTM_LR = 1e-3
LSTM_SEEDS = [0, 1, 2]        # báo cáo: lặp với nhiều seed
N_JOBS = -1                   # -1: tất cả lõi CPU (LightGBM, torch CPU threads)
SEED = 42
LGB_OBJECTIVE = "regression"  # L2 như báo cáo; thử "tweedie"
LGB_MAX_ROUNDS = 3000
LGB_EARLY_STOP = 100
LGB_ORIGIN_STRIDE = 1         # bước giữa các mốc dự báo khi tạo mẫu LightGBM (1: mọi ngày)
REFIT_FULL = True             # huấn luyện lại trên 90 ngày sau khi chọn số vòng/epoch (mục 6.5)
MASK_K = 2                    # E3m: bỏ ngày có > k giờ hết hàng (6–22)
PPLUS_MIN = 0.3               # ngưỡng P+ cho fallback khôi phục
PROFILE_SHRINK_K = 5.0        # co hồ sơ giờ của chuỗi về cửa hàng × danh mục cấp 3
SIGMA_SHRINK_K = 7.0          # co sigma_e của chuỗi về nhóm
SERVICE_LEVELS = [0.80, 0.90, 0.95]
RUN_LSTM = True
RUN_SHAP = True
HF_REPO = "Dingdong-Inc/FreshRetailNet-50K"
HF_REVISION = "08c1fab7f9257bc73679d415d65d644165d351d4"   # commit đã kiểm tra 03/10/2026
DATA_SOURCE = "hf"            # "hf": tải từ Hugging Face, lỗi thì dùng LOCAL_DATA_DIR; "local": chỉ dùng LOCAL_DATA_DIR
LOCAL_DATA_DIR = "../sources" # chứa train.parquet, eval.parquet
OUTPUT_DIR = None             # None -> outputs_<n_series>

## 1. Môi trường, seed, đo thời gian và RAM

Ô dưới đặt seed, chọn thiết bị và tạo hàm đo thời gian, RAM cho từng bước.

In [ ]:
import os, sys, time, json, math, random, resource, warnings, platform
from contextlib import contextmanager
import numpy as np
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import matplotlib
import matplotlib.pyplot as plt
import psutil
warnings.filterwarnings("ignore", category=UserWarning)

if isinstance(LSTM_SEEDS, str): LSTM_SEEDS = json.loads(LSTM_SEEDS)          # papermill -p truyền chuỗi
if isinstance(SERVICE_LEVELS, str): SERVICE_LEVELS = json.loads(SERVICE_LEVELS)
if isinstance(SUBSET_N_SERIES, str): SUBSET_N_SERIES = None if SUBSET_N_SERIES.lower() in ("none", "") else int(SUBSET_N_SERIES)
N_CPU = os.cpu_count() if N_JOBS in (None, -1) else int(N_JOBS)
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except ImportError:
        pass
seed_everything(SEED)

TIMINGS, RAM = {}, {}
_proc = psutil.Process()
def peak_rss_gb():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r / (1024**3 if sys.platform == "darwin" else 1024**2)
import threading
class _RssSampler(threading.Thread):
    # lấy mẫu RSS mỗi 0.2 s để biết đỉnh RAM riêng của từng bước (ru_maxrss chỉ cho đỉnh tích lũy)
    def __init__(s):
        super().__init__(daemon=True); s.max = 0; s.stop = False
    def run(s):
        while not s.stop:
            s.max = max(s.max, _proc.memory_info().rss); time.sleep(0.2)
@contextmanager
def stage(name):
    t0 = time.perf_counter(); smp = _RssSampler(); smp.start()
    yield
    smp.stop = True; smp.join()
    TIMINGS[name] = TIMINGS.get(name, 0.0) + time.perf_counter() - t0
    st_max = max(smp.max, _proc.memory_info().rss) / 1024**3
    RAM[name] = {"rss_gb": _proc.memory_info().rss / 1024**3,
                 "stage_peak_rss_gb": max(st_max, RAM.get(name, {}).get("stage_peak_rss_gb", 0)),
                 "cum_peak_rss_gb": peak_rss_gb()}
    print(f"[{name}] {TIMINGS[name]:.1f}s | RSS {RAM[name]['rss_gb']:.2f} GB | stage peak {RAM[name]['stage_peak_rss_gb']:.2f} GB"
          f" | cumulative peak {RAM[name]['cum_peak_rss_gb']:.2f} GB")

print(platform.platform(), "| python", sys.version.split()[0], "| CPU threads used:", N_CPU,
      "| RAM total %.1f GB" % (psutil.virtual_memory().total / 1024**3))

## 2. Tải dữ liệu

Dữ liệu tải từ Hugging Face. Nếu tải lỗi, hoặc khi DATA_SOURCE = "local", notebook dùng bản cục bộ trong LOCAL_DATA_DIR.

In [ ]:
def locate_files():
    local = {s: os.path.join(LOCAL_DATA_DIR, f"{s}.parquet") for s in ("train", "eval")}
    if DATA_SOURCE == "hf":
        try:
            from huggingface_hub import hf_hub_download
            return {s: hf_hub_download(HF_REPO, f"data/{s}.parquet", repo_type="dataset",
                                       revision=HF_REVISION) for s in ("train", "eval")}, "huggingface"
        except Exception as e:
            print("HF download failed -> local fallback:", repr(e)[:200])
    assert all(os.path.exists(p) for p in local.values()), f"Không tìm thấy dữ liệu cục bộ: {local}"
    return local, "local"

KEY_COLS = ["store_id", "product_id", "dt", "city_id", "management_group_id", "first_category_id",
            "second_category_id", "third_category_id", "stock_hour6_22_cnt"]
with stage("load_data"):
    FILES, SRC = locate_files()
    print("source:", SRC, FILES)
    # đọc theo cột để giới hạn RAM: trước hết chỉ các cột khóa/phân cấp, các cột còn lại đọc từng cột ở bước 4
    tr = pq.read_table(FILES["train"], columns=KEY_COLS); ev = pq.read_table(FILES["eval"], columns=KEY_COLS)
    print("train rows", tr.num_rows, "| eval rows", ev.num_rows)
    assert tr.num_rows == 4_500_000 and ev.num_rows == 350_000

## 3. Chọn tập con phân tầng

Tập con được lấy mẫu phân tầng theo thành phố, nhóm quản lý và mức hết hàng (báo cáo mục 6.8). Mọi mô hình dùng cùng tập con.

In [ ]:
with stage("subset"):
    key_cols = ["store_id", "product_id", "city_id", "management_group_id"]
    sc = tr.select(key_cols + ["stock_hour6_22_cnt"]).to_pandas()
    ser = sc.groupby(["store_id", "product_id"], sort=True).agg(
        city_id=("city_id", "first"), mg=("management_group_id", "first"),
        so_rate=("stock_hour6_22_cnt", "mean")).reset_index()
    ser["so_level"] = pd.qcut(ser["so_rate"].rank(method="first"), 3, labels=False)
    if SUBSET_N_SERIES is None or SUBSET_N_SERIES >= len(ser):
        chosen = ser
    else:
        frac = SUBSET_N_SERIES / len(ser)
        rng = np.random.default_rng(SEED)
        parts = []
        for _, g in ser.groupby(["city_id", "mg", "so_level"]):
            k = frac * len(g); k = int(np.floor(k) + (rng.random() < k - np.floor(k)))
            if k > 0: parts.append(g.sample(n=min(k, len(g)), random_state=int(rng.integers(1e9))))
        chosen = pd.concat(parts)
    chosen = chosen.sort_values(["store_id", "product_id"]).reset_index(drop=True)
    N = len(chosen)
    print("series used:", N)
    OUT = OUTPUT_DIR or f"outputs_{N}"
    os.makedirs(OUT, exist_ok=True)
    del sc

## 4. Mảng ngày N × 97 và mảng giờ N × 97 × 24

Chỉ số ngày:
- 0–82: huấn luyện (28/03–18/06).
- 83–89: validation (19/06–25/06).
- 90–96: kiểm tra, chính là tập eval (26/06–02/07).

In [ ]:
def row_order(tbl, chosen):
    # thứ tự dòng (chuỗi, ngày) của tập con trong tệp parquet
    k = tbl.select(["store_id", "product_id", "dt"]).to_pandas()
    k["row"] = np.arange(len(k))
    k = k.merge(chosen[["store_id", "product_id"]].assign(sidx=np.arange(len(chosen))), on=["store_id", "product_id"])
    return pa.array(k.sort_values(["sidx", "dt"])["row"].values)

with stage("build_panel"):
    T_TR, T_EV = 90, 7
    T_ALL = T_TR + T_EV
    ROWS = {"train": row_order(tr, chosen), "eval": row_order(ev, chosen)}
    assert len(ROWS["train"]) == N * T_TR and len(ROWS["eval"]) == N * T_EV
    def col(split, c):
        if c in KEY_COLS:
            src = tr if split == "train" else ev
            return src.column(c).take(ROWS[split])
        return pq.read_table(FILES[split], columns=[c]).column(c).take(ROWS[split])
    def daily(c, dtype=np.float32):
        a = col("train", c).to_numpy().reshape(N, T_TR); b = col("eval", c).to_numpy().reshape(N, T_EV)
        return np.concatenate([a, b], 1).astype(dtype)
    def hourly(c, dtype):
        a = pc.list_flatten(col("train", c).combine_chunks()).to_numpy().astype(dtype).reshape(N, T_TR, 24)
        b = pc.list_flatten(col("eval", c).combine_chunks()).to_numpy().astype(dtype).reshape(N, T_EV, 24)
        return np.concatenate([a, b], 1)
    Y = daily("sale_amount")
    HS = hourly("hours_sale", np.float32)
    ST = hourly("hours_stock_status", np.int8)           # 1 = hết hàng
    CNT = daily("stock_hour6_22_cnt", np.int16)
    DISC = daily("discount"); ACT = daily("activity_flag"); HOL = daily("holiday_flag")
    WEATHER_COLS = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
    WEA = np.stack([daily(c) for c in WEATHER_COLS], -1)  # N × 97 × 4
    dates = pd.to_datetime(np.concatenate([col("train", "dt").to_numpy()[:T_TR], col("eval", "dt").to_numpy()[:T_EV]]).astype(str))
    DOW = dates.dayofweek.values.astype(np.int8); DOM = dates.day.values.astype(np.int8)
    STATIC_COLS = ["city_id", "store_id", "management_group_id", "first_category_id",
                   "second_category_id", "third_category_id", "product_id"]
    STATIC = {c: col("train", c).to_numpy().reshape(N, T_TR)[:, 0].astype(np.int32) for c in STATIC_COLS}
    STATIC_CARD = {c: int(pc.max(tr.column(c)).as_py()) + 1 for c in STATIC_COLS}   # từ toàn bộ dữ liệu
    TR_END, VA_END = 83, 90      # [0,83) train, [83,90) val, [90,97) test
    del tr, ev; pa.default_memory_pool().release_unused()
    print("dates:", dates[0].date(), "->", dates[TR_END-1].date(), "| val", dates[TR_END].date(), "->",
          dates[VA_END-1].date(), "| test", dates[VA_END].date(), "->", dates[-1].date())
    print("RSS after freeing arrow tables: %.2f GB" % (_proc.memory_info().rss / 1024**3))

### 4.1 Kiểm tra nhất quán và tiền xử lý

Kiểm tra theo báo cáo mục 6.3. Tiền xử lý theo mục 6.6.

In [ ]:
with stage("checks_preprocess"):
    chk = {
        "max |sum(hours_sale) - sale_amount|": float(np.abs(HS.sum(-1) - Y).max()),
        "cnt == sum(status[6:22]) share": float((ST[:, :, 6:22].sum(-1) == CNT).mean()),
        "stockout hour share 6-22 (train days)": float(ST[:, :VA_END, 6:22].mean()),
        "stockout hour share 6-22 (test days)": float(ST[:, VA_END:, 6:22].mean()),
        "days with >=1 stockout hour (train)": float((CNT[:, :VA_END] > 0).mean()),
        "days fully out (16h) (train)": float((CNT[:, :VA_END] == 16).mean()),
        "discount == 0 share": float((DISC == 0).mean()),
        "discount > 1 share": float((DISC > 1).mean()),
    }
    for k, v in chk.items(): print(f"{k:45s} {v:.6g}")
    # discount = 0 -> thiếu, điền trung vị theo sản phẩm × activity_flag (trên dữ liệu train); > 1 -> 1
    DISC = np.where(DISC == 0, np.nan, np.minimum(DISC, 1.0)).astype(np.float32)
    dfd = pd.DataFrame({"p": np.repeat(STATIC["product_id"], T_ALL), "a": ACT.ravel(), "d": DISC.ravel(),
                        "tr": np.tile(np.arange(T_ALL) < VA_END, N)})
    med = dfd[dfd.tr].groupby(["p", "a"], as_index=False)["d"].median().rename(columns={"d": "med"})
    fill = dfd[["p", "a"]].merge(med, on=["p", "a"], how="left")["med"].values.astype(np.float32)
    DISC = np.where(np.isnan(DISC), fill.reshape(N, T_ALL), DISC)
    DISC = np.where(np.isnan(DISC), 1.0, DISC).astype(np.float32)
    del dfd, fill
    json.dump(chk, open(f"{OUT}/data_checks.json", "w"), indent=1)

## 5. Xử lý giờ hết hàng: thô, che, khôi phục

Theo báo cáo mục 6.7.
- Hồ sơ giờ p(i,h) là tỷ trọng doanh số theo giờ. Hồ sơ được ước lượng trên các ngày còn hàng đủ 24 giờ của chuỗi trong tập huấn luyện (ngày 0–82). Ngày thường và cuối tuần tách riêng. Hồ sơ được co về hồ sơ của cửa hàng × danh mục cấp 3.
- Nhu cầu khôi phục D̃ = Y⁺ / P⁺. Giờ hết hàng nhận max(y, D̃·p).
- Khi P⁺ < 0.3, dùng trung bình nhu cầu cùng thứ của các tuần trước.
- Giá trị khôi phục được chặn trên bởi phân vị 99 của chuỗi.

In [ ]:
WEEKEND = (DOW >= 5).astype(np.int8)       # 0 = ngày thường, 1 = cuối tuần

def estimate_profiles(HS, ST, day_mask):
    full24 = (ST.sum(-1) == 0) & day_mask[None, :]                   # N × T
    P = np.zeros((N, 2, 24), np.float32)
    S_i = np.zeros((N, 2, 24), np.float64); n_i = np.zeros((N, 2))
    for w in (0, 1):
        m = full24 & (WEEKEND[None, :] == w)
        S_i[:, w] = (HS * m[..., None]).sum(1)
        n_i[:, w] = m.sum(1)
    tot_glob = S_i.sum((0))                                          # 2 × 24
    p_glob = tot_glob / tot_glob.sum(-1, keepdims=True)
    grp = pd.factorize(pd.Series(STATIC["store_id"]).astype(str) + "_" + pd.Series(STATIC["third_category_id"]).astype(str))[0]
    G = grp.max() + 1
    S_g = np.zeros((G, 2, 24)); np.add.at(S_g, grp, S_i)
    tg = S_g.sum(-1, keepdims=True)
    p_g = np.where(tg > 0, S_g / np.maximum(tg, 1e-12), p_glob[None])
    ti = S_i.sum(-1, keepdims=True)
    p_i = np.where(ti > 0, S_i / np.maximum(ti, 1e-12), p_g[grp])
    nn = np.where(ti[..., 0] > 0, n_i, 0.0)[..., None]
    P = (nn * p_i + PROFILE_SHRINK_K * p_g[grp]) / (nn + PROFILE_SHRINK_K)
    return (P / P.sum(-1, keepdims=True)).astype(np.float32), n_i

def recover(HS, ST, Y, PROF, cap_days):
    p_day = PROF[:, WEEKEND, :]                                      # N × T × 24
    instock = 1 - ST
    YP = (HS * instock).sum(-1)                                      # Y+
    PP = (p_day * instock).sum(-1)                                   # P+
    any_so = ST.sum(-1) > 0
    base = np.where(PP >= PPLUS_MIN, YP / np.maximum(PP, 1e-6), np.nan)
    hourly_fill = np.maximum(HS, base[..., None] * p_day) * ST       # giờ hết hàng: max(y, D~·p)
    D = np.where(any_so, YP + hourly_fill.sum(-1), Y)
    fb = any_so & (PP < PPLUS_MIN)
    D = np.where(fb, np.nan, D)
    # fallback: trung bình nhu cầu cùng thứ các tuần trước (chỉ giá trị hợp lệ), nếu không có -> trung bình hợp lệ trước đó, cuối cùng -> Y
    for t in range(D.shape[1]):
        need = np.isnan(D[:, t])
        if not need.any(): continue
        prev = D[:, t - 7::-7] if t >= 7 else np.empty((N, 0))
        m1 = np.nanmean(np.where(np.isnan(prev), np.nan, prev), 1) if prev.shape[1] else np.full(N, np.nan)
        m2 = np.nanmean(D[:, :t], 1) if t > 0 else np.full(N, np.nan)
        val = np.where(np.isnan(m1), m2, m1)
        val = np.where(np.isnan(val), Y[:, t], val)
        D[need, t] = np.maximum(val[need], Y[need, t])
    cap = np.nanpercentile(np.where(fb[:, :cap_days], np.nan, D[:, :cap_days]), 99, axis=1)
    D = np.minimum(D, np.maximum(cap[:, None], Y))
    return D.astype(np.float32), YP.astype(np.float32), PP.astype(np.float32), fb

with stage("demand_recovery"):
    import warnings as _w; _w.simplefilter("ignore", RuntimeWarning)
    train_days = np.arange(T_ALL) < TR_END
    PROF, n_prof = estimate_profiles(HS, ST, train_days)
    DREC, YPLUS, PPLUS, FB = recover(HS, ST, Y, PROF, TR_END)
    rec_stats = {
        "share series-days recovered (any stockout hour, days 0-89)": float((ST[:, :VA_END].sum(-1) > 0).mean()),
        "share fallback P+<thr (days 0-89)": float(FB[:, :VA_END].mean()),
        "sum recovered / sum observed (days 0-89)": float(DREC[:, :VA_END].sum() / Y[:, :VA_END].sum()),
        "median #full-24h profile days per series (weekday, weekend)": [float(np.median(n_prof[:, 0])), float(np.median(n_prof[:, 1]))],
    }
    print(json.dumps(rec_stats, indent=1))

### 5.1 Kiểm chứng khôi phục bằng che nhân tạo

Cách làm giống bài báo gốc. Chọn các ngày còn hàng đủ 24 giờ trong tập huấn luyện. Che từ giờ s ∈ [10, 21] đến hết ngày, vì hết hàng cuối ngày là dạng phổ biến. Sau đó khôi phục và so với doanh số thật.

Lưu ý: hồ sơ giờ được ước lượng có dùng chính các ngày này. Vì vậy kết quả hơi lạc quan.

In [ ]:
with stage("recovery_validation"):
    rng = np.random.default_rng(SEED)
    full24 = (ST.sum(-1) == 0) & (Y > 0) & train_days[None, :]
    ii, tt = np.nonzero(full24)
    sel = rng.choice(len(ii), size=min(20000, len(ii)), replace=False)
    ii, tt = ii[sel], tt[sel]
    s = rng.integers(10, 22, size=len(ii))
    st_art = (np.arange(24)[None, :] >= s[:, None]).astype(np.int8)
    hs = HS[ii, tt] * (1 - st_art)                 # doanh số giờ bị che coi như mất (như khi hết hàng thật)
    p = PROF[ii, WEEKEND[tt]]
    yp = hs.sum(-1); pp = (p * (1 - st_art)).sum(-1)
    ok = pp >= PPLUS_MIN
    drec = np.where(ok, yp + (np.maximum(hs, (yp / np.maximum(pp, 1e-6))[:, None] * p) * st_art).sum(-1), np.nan)
    truth = Y[ii, tt]
    def wape(a, b): return float(np.abs(a - b).sum() / b.sum())
    def wpe(a, b): return float((a - b).sum() / b.sum())
    recval = pd.DataFrame({
        "method": ["censored (Y+ only)", "recovered (D~)"],
        "WAPE": [wape(yp[ok], truth[ok]), wape(drec[ok], truth[ok])],
        "WPE": [wpe(yp[ok], truth[ok]), wpe(drec[ok], truth[ok])],
        "n_days": [int(ok.sum())] * 2})
    recval.to_csv(f"{OUT}/recovery_validation.csv", index=False)
    display(recval.round(4))

## 6. Đánh giá trên giờ còn hàng và các hàm dùng chung

Theo báo cáo mục 7.5. Trên ngày kiểm tra:
- Y⁺ là doanh số trong các giờ còn hàng.
- D̂⁺ = D̂ · P⁺, với P⁺ lấy từ hồ sơ giờ ước lượng trên tập huấn luyện.

Kiểm tra độ vững: chỉ dùng các ngày còn hàng đủ 16 giờ (stock_hour6_22_cnt = 0).

In [ ]:
TEST = slice(VA_END, T_ALL); VAL = slice(TR_END, VA_END)
FULL16_TEST = CNT[:, TEST] == 0
def forecast_metrics(pred, mask=None):
    # pred: N × 7 dự báo nhu cầu ngày cho ngày kiểm tra
    dp = pred * PPLUS[:, TEST]; yp = YPLUS[:, TEST]
    if mask is not None: dp, yp = dp[mask], yp[mask]
    e = dp - yp
    return {"WAPE": float(np.abs(e).sum() / yp.sum()), "WPE": float(e.sum() / yp.sum()),
            "MAE": float(np.abs(e).mean()), "RMSE": float(np.sqrt((e ** 2).mean()))}
PRED_TEST, PRED_VAL, EXTRA = {}, {}, {}
def register(name, pred_val, pred_test):
    PRED_VAL[name] = np.maximum(pred_val, 0).astype(np.float32)
    PRED_TEST[name] = np.maximum(pred_test, 0).astype(np.float32)
    m = forecast_metrics(PRED_TEST[name]); r = forecast_metrics(PRED_TEST[name], FULL16_TEST)
    print(f"{name:5s} in-stock hours: " + " ".join(f"{k}={v:.4f}" for k, v in m.items())
          + " | full-16h days: " + " ".join(f"{k}={v:.4f}" for k, v in r.items()))

## 7. E1: Seasonal Naive theo tuần

D̂(T+h) = Y(T+h−7), tính trên doanh số thô.

In [ ]:
with stage("E1_seasonal_naive"):
    register("E1", Y[:, TR_END - 7:TR_END], Y[:, VA_END - 7:VA_END])

## 8. LightGBM (E2, E3, E3m, A1, A2)

- Mô hình toàn cục, dự báo trực tiếp với biến tầm h.
- Đặc trưng trễ: y(T+h−7), y(T+h−14) và trung bình trượt 7/14/28 ngày tại mốc T.
- Biến biết trước của ngày đích: khuyến mãi, lịch, thời tiết.
- Hàm mất mát L2. Early stopping trên validation, mốc T = 18/06.
- Sau đó huấn luyện lại trên 90 ngày với số vòng tốt nhất. Dự báo từ mốc T = 25/06.

In [ ]:
import lightgbm as lgb
H = np.arange(1, 8)
# đặc trưng hết hàng theo ngày (dùng ở dạng trễ)
so_hours = np.where(ST[:, :, 6:22] == 1, np.arange(6, 22)[None, None, :], 99).min(-1)
FIRST_SO = np.where(so_hours == 99, 24, so_hours).astype(np.float32)              # giờ hết hàng đầu tiên (6–21), 24 = không
before = (np.arange(24)[None, None, :] < FIRST_SO[..., None])
SALES_BEFORE_SO = (HS * before).sum(-1).astype(np.float32)
CNT_F = CNT.astype(np.float32)

def roll_mean(A, k):
    cs = np.concatenate([np.zeros((A.shape[0], 1)), np.cumsum(A, 1, dtype=np.float64)], 1)
    out = np.full(A.shape, np.nan, np.float32)
    out[:, k - 1:] = (cs[:, k:] - cs[:, :-k]) / k
    return out

def lgb_frame(D, origins, stockout_feats, weather, promo, target=None, mask_k=None):
    # D: N × 97 chuỗi nhu cầu dùng cho trễ; origins: các mốc T (chỉ số ngày quan sát cuối)
    i = np.repeat(np.arange(N), len(origins) * 7)
    T = np.tile(np.repeat(origins, 7), N)
    h = np.tile(np.tile(H, len(origins)), N)
    d = T + h
    rm7, rm14, rm28 = roll_mean(D, 7), roll_mean(D, 14), roll_mean(D, 28)
    sd7 = np.sqrt(np.maximum(roll_mean(D ** 2, 7) - rm7 ** 2, 0))
    f = {"h": h.astype(np.int8), "lag_dow1": D[i, d - 7], "lag_dow2": D[i, d - 14], "last": D[i, T],
         "rm7": rm7[i, T], "rm14": rm14[i, T], "rm28": rm28[i, T], "sd7": sd7[i, T],
         "dow": DOW[d], "dom": DOM[d], "holiday_flag": HOL[i, d]}
    if promo:
        f["discount"] = DISC[i, d]; f["activity_flag"] = ACT[i, d]
        f["disc_mean7"] = roll_mean(DISC, 7)[i, T]
    if weather:
        for k, c in enumerate(WEATHER_COLS): f[c] = WEA[i, d, k]
    if stockout_feats:
        cm7 = roll_mean(CNT_F, 7); pp7 = roll_mean(PPLUS, 7); any7 = roll_mean((CNT_F > 0).astype(np.float32), 7)
        f.update({"so_cnt_last": CNT_F[i, T], "so_cnt_mean7": cm7[i, T], "so_cnt_lag_dow1": CNT_F[i, d - 7],
                  "so_days7": any7[i, T], "first_so_hour_last": FIRST_SO[i, T],
                  "sales_before_so_last": SALES_BEFORE_SO[i, T], "pplus_last": PPLUS[i, T], "pplus_mean7": pp7[i, T]})
    for c in STATIC_COLS: f[c] = STATIC[c][i]
    # ma trận float32 liền khối (tránh bản sao DataFrame và việc LightGBM ép sang float64)
    names = list(f)
    X = np.empty((len(i), len(names)), np.float32)
    for k, c in enumerate(names):
        X[:, k] = f[c]; f[c] = None
    del f
    y = None if target is None else target[i, d]
    w = None if mask_k is None else (CNT[i, d] <= mask_k).astype(np.float32)
    return X, y, w, names

LGB_PARAMS = dict(objective=LGB_OBJECTIVE, learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  max_cat_to_onehot=8, cat_smooth=20, num_threads=N_CPU, seed=SEED, verbose=-1,
                  deterministic=True, force_row_wise=True)
TR_ORIG = np.arange(27, TR_END - 7, LGB_ORIGIN_STRIDE)     # mục tiêu ≤ ngày 82
FULL_ORIG = np.arange(27, VA_END - 7, LGB_ORIGIN_STRIDE)   # mục tiêu ≤ ngày 89
LGB_INFO = {}

def run_lgb(name, D, target, stockout_feats, weather=True, promo=True, mask_k=None):
    with stage(name):
        Xtr, ytr, wtr, names = lgb_frame(D, TR_ORIG, stockout_feats, weather, promo, target, mask_k)
        Xva, yva, wva, _ = lgb_frame(D, np.array([TR_END - 1]), stockout_feats, weather, promo, target, mask_k)
        cats = STATIC_COLS
        dtr = lgb.Dataset(Xtr, ytr, weight=wtr, feature_name=names, categorical_feature=cats, free_raw_data=True)
        dva = lgb.Dataset(Xva, yva, weight=wva, feature_name=names, categorical_feature=cats, reference=dtr)
        m1 = lgb.train(LGB_PARAMS, dtr, LGB_MAX_ROUNDS, valid_sets=[dva],
                       callbacks=[lgb.early_stopping(LGB_EARLY_STOP, verbose=False)])
        best = m1.best_iteration or LGB_MAX_ROUNDS
        pred_val = m1.predict(Xva, num_iteration=best).reshape(N, 7)
        n_train_rows = len(Xtr); del Xtr, dtr
        if REFIT_FULL:
            Xf, yf, wf, _ = lgb_frame(D, FULL_ORIG, stockout_feats, weather, promo, target, mask_k)
            m2 = lgb.train(LGB_PARAMS, lgb.Dataset(Xf, yf, weight=wf, feature_name=names, categorical_feature=cats), best)
            del Xf
        else:
            m2 = m1
        Xte, _, _, _ = lgb_frame(D, np.array([VA_END - 1]), stockout_feats, weather, promo)
        Xte = pd.DataFrame(Xte, columns=names)
        pred_test = m2.predict(Xte, num_iteration=m2.best_iteration if m2 is m1 else None).reshape(N, 7)
        LGB_INFO[name] = {"best_iter": int(best), "n_train_rows": int(n_train_rows), "n_features": Xte.shape[1],
                          "model": m2, "X_test": Xte}
        register(name, pred_val, pred_test)
        print(f"   best_iter={best}, train rows={n_train_rows:,}, features={Xte.shape[1]}")

run_lgb("E2", Y, Y, stockout_feats=False)
run_lgb("E3", DREC, DREC, stockout_feats=True)
run_lgb("E3m", Y, Y, stockout_feats=True, mask_k=MASK_K)
run_lgb("A1", DREC, DREC, stockout_feats=True, weather=False)
run_lgb("A2", DREC, DREC, stockout_feats=True, promo=False)

## 9. LSTM (E4 thô, E5 khôi phục)

- Cửa sổ quá khứ L = 28 ngày, đầu ra 7 ngày, trượt bước 1. Mỗi chuỗi có 49 mẫu trên 83 ngày huấn luyện.
- Có nhánh biến biết trước cho 7 ngày đích và embedding phân cấp.
- Nhu cầu được chia cho trung bình của chuỗi trên tập huấn luyện. Thời tiết được chuẩn hóa bằng thống kê tập huấn luyện.
- Hàm mất mát MSE, tối ưu Adam, early stopping theo WAPE validation.
- Lặp với nhiều seed.

In [ ]:
if RUN_LSTM:
    import torch, torch.nn as nn
    torch.set_num_threads(N_CPU)
    if DEVICE == "auto":
        DEV = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
    else:
        DEV = DEVICE
    print("torch", torch.__version__, "| device:", DEV)
    L, HOR = 28, 7
    wmu = WEA[:, :TR_END].reshape(-1, 4).mean(0); wsd = WEA[:, :TR_END].reshape(-1, 4).std(0) + 1e-6
    WEA_Z = ((WEA - wmu) / wsd).astype(np.float32)
    DOW_OH = np.eye(7, dtype=np.float32)[DOW]                                  # 97 × 7
    FUT = np.concatenate([DISC[..., None], ACT[..., None], HOL[..., None],
                          np.broadcast_to(DOW_OH, (N, T_ALL, 7)), WEA_Z], -1).astype(np.float32)   # N × 97 × 14
    STAT = np.stack([STATIC[c] for c in STATIC_COLS], 1).astype(np.int64)

    class Net(nn.Module):
        def __init__(s, f_past, f_fut):
            super().__init__()
            dims = {"city_id": 4, "store_id": 16, "management_group_id": 3, "first_category_id": 4,
                    "second_category_id": 6, "third_category_id": 8, "product_id": 16}
            s.embs = nn.ModuleList([nn.Embedding(STATIC_CARD[c], dims[c]) for c in STATIC_COLS])
            s.lstm = nn.LSTM(f_past, LSTM_HIDDEN, num_layers=LSTM_LAYERS, batch_first=True,
                             dropout=0.1 if LSTM_LAYERS > 1 else 0.0)
            s.fut = nn.Sequential(nn.Linear(HOR * f_fut, 32), nn.ReLU())
            s.head = nn.Sequential(nn.Linear(LSTM_HIDDEN + 32 + sum(dims.values()), 64), nn.ReLU(), nn.Linear(64, HOR))
        def forward(s, xp, xf, xs):
            _, (h, _) = s.lstm(xp)
            e = torch.cat([emb(xs[:, k]) for k, emb in enumerate(s.embs)], 1)
            return s.head(torch.cat([h[-1], s.fut(xf.flatten(1)), e], 1))

    def make_past(Dn, stockout_feats):
        parts = [Dn[..., None]]
        if stockout_feats: parts.append((CNT_F / 16.0)[..., None])
        parts += [DISC[..., None], ACT[..., None], HOL[..., None], np.broadcast_to(DOW_OH, (N, T_ALL, 7)), WEA_Z]
        return np.concatenate(parts, -1).astype(np.float32)

    def gather(P, F, Dn, i, s):
        tp = s[:, None] + np.arange(L)[None, :]; tf = s[:, None] + L + np.arange(HOR)[None, :]
        return P[i[:, None], tp], F[i[:, None], tf], Dn[i[:, None], tf]

    def predict(model, P, F, origin_last):       # dự báo cho các ngày origin_last+1 … +7
        model.eval(); s = np.full(N, origin_last - L + 1); out = []
        with torch.no_grad():
            for b in range(0, N, 8192):
                ib = np.arange(b, min(b + 8192, N))
                xp, xf, _ = gather(P, F, P[..., 0], ib, s[ib])
                out.append(model(torch.from_numpy(xp).to(DEV), torch.from_numpy(xf).to(DEV),
                                 torch.from_numpy(STAT[ib]).to(DEV)).float().cpu().numpy())
        return np.concatenate(out)

    def train_lstm(D, stockout_feats, seed, last_target_day, epochs, val=True):
        seed_everything(seed)
        scale = np.maximum(D[:, :TR_END].mean(1), 0.01).astype(np.float32)[:, None]
        Dn = (D / scale).astype(np.float32)
        P = make_past(Dn, stockout_feats)
        model = Net(P.shape[-1], FUT.shape[-1]).to(DEV)
        opt = torch.optim.Adam(model.parameters(), lr=LSTM_LR)
        starts = np.arange(0, last_target_day - L - HOR + 2)           # mục tiêu ≤ last_target_day
        ii, ss = np.repeat(np.arange(N), len(starts)), np.tile(starts, N)
        rng = np.random.default_rng(seed); hist = []; best = (np.inf, 0, None)
        for ep in range(epochs):
            model.train(); perm = rng.permutation(len(ii)); tl = 0.0
            for b in range(0, len(perm), LSTM_BATCH_SIZE):
                j = perm[b:b + LSTM_BATCH_SIZE]
                xp, xf, yt = gather(P, FUT, Dn, ii[j], ss[j])
                pred = model(torch.from_numpy(xp).to(DEV), torch.from_numpy(xf).to(DEV), torch.from_numpy(STAT[ii[j]]).to(DEV))
                loss = ((pred - torch.from_numpy(yt).to(DEV)) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item() * len(j)
            rec = {"epoch": ep + 1, "train_mse": tl / len(perm)}
            if val:
                pv = predict(model, P, FUT, TR_END - 1) * scale
                rec["val_wape"] = float(np.abs(pv - D[:, VAL]).sum() / D[:, VAL].sum())
                if rec["val_wape"] < best[0]:
                    best = (rec["val_wape"], ep + 1, {k: v.detach().clone() for k, v in model.state_dict().items()})
                elif ep + 1 - best[1] >= LSTM_PATIENCE:
                    hist.append(rec); break
            hist.append(rec)
        if val: model.load_state_dict(best[2])
        return model, P, scale, hist, (best[1] if val else epochs), len(ii)

    LSTM_INFO = {}
    def run_lstm(name, D, stockout_feats):
        with stage(name):
            pv_all, pt_all, per_seed = [], [], []
            for sd in LSTM_SEEDS:
                t0 = time.perf_counter()
                m1, P1, sc, hist, best_ep, n_samples = train_lstm(D, stockout_feats, sd, TR_END - 1, LSTM_EPOCHS)
                pv = predict(m1, P1, FUT, TR_END - 1) * sc
                if REFIT_FULL:
                    m2, P2, sc2, _, _, _ = train_lstm(D, stockout_feats, sd, VA_END - 1, best_ep, val=False)
                else:
                    m2, P2, sc2 = m1, P1, sc
                pt = predict(m2, P2, FUT, VA_END - 1) * sc2
                pv_all.append(pv); pt_all.append(pt)
                per_seed.append({"seed": sd, "best_epoch": best_ep, "train_samples": n_samples,
                                 "sec": time.perf_counter() - t0, **forecast_metrics(np.maximum(pt, 0))})
                print(f"   seed {sd}: best_epoch={best_ep}, samples/epoch={n_samples:,}, {per_seed[-1]['sec']:.0f}s, "
                      f"WAPE={per_seed[-1]['WAPE']:.4f}")
            LSTM_INFO[name] = {"per_seed": pd.DataFrame(per_seed), "model": m2, "P": P2, "scale": sc2, "hist": hist}
            register(name, np.mean(pv_all, 0), np.mean(pt_all, 0))     # dự báo = trung bình các seed
    run_lstm("E4", Y, stockout_feats=False)
    run_lstm("E5", DREC, stockout_feats=True)

## 10. Bảng kết quả dự báo và kiểm tra độ vững

Sai số tính trên giờ còn hàng.

In [ ]:
rows = []
for name in PRED_TEST:
    m = forecast_metrics(PRED_TEST[name]); r = forecast_metrics(PRED_TEST[name], FULL16_TEST)
    row = {"exp": name, **m, **{f"{k}_full16": v for k, v in r.items()}}
    if RUN_LSTM and name in LSTM_INFO and len(LSTM_SEEDS) > 1:
        ps = LSTM_INFO[name]["per_seed"]; row["WAPE_seed_mean"] = ps.WAPE.mean(); row["WAPE_seed_std"] = ps.WAPE.std()
    rows.append(row)
RES = pd.DataFrame(rows).set_index("exp")
RES.to_csv(f"{OUT}/forecast_metrics.csv")
print(f"N series = {N}; test days = 7; series-days = {N*7:,}; full-16h test days = {int(FULL16_TEST.sum()):,}")
display(RES.round(4))

### 10.1 Theo tầm dự báo và theo nhóm

Theo báo cáo mục 7.6.

In [ ]:
def wape_mask(pred, mask):
    dp = (pred * PPLUS[:, TEST])[mask]; yp = YPLUS[:, TEST][mask]; return float(np.abs(dp - yp).sum() / yp.sum())
hz = pd.DataFrame({n: [wape_mask(p, np.broadcast_to(np.arange(7)[None, :] == k, (N, 7))) for k in range(7)]
                   for n, p in PRED_TEST.items()}, index=pd.Index(range(1, 8), name="h"))
hz.to_csv(f"{OUT}/wape_by_horizon.csv")
mean_tr = Y[:, :VA_END].mean(1); so_tr = CNT[:, :VA_END].mean(1)
groups = {
    "high sales (>= median)": np.broadcast_to((mean_tr >= np.median(mean_tr))[:, None], (N, 7)),
    "low sales (< median)": np.broadcast_to((mean_tr < np.median(mean_tr))[:, None], (N, 7)),
    "high stockout series (top tercile)": np.broadcast_to((so_tr >= np.quantile(so_tr, 2/3))[:, None], (N, 7)),
    "promo day (activity_flag=1)": ACT[:, TEST] == 1,
    "non-promo day": ACT[:, TEST] == 0,
    "holiday_flag=1": HOL[:, TEST] == 1,
    "holiday_flag=0": HOL[:, TEST] == 0,
}
for g in np.unique(STATIC["management_group_id"]):
    groups[f"management_group {g}"] = np.broadcast_to((STATIC["management_group_id"] == g)[:, None], (N, 7))
grp_tab = pd.DataFrame({n: {g: wape_mask(p, m) for g, m in groups.items()} for n, p in PRED_TEST.items()})
grp_tab["n_series_days"] = [int(m.sum()) for m in groups.values()]
grp_tab.to_csv(f"{OUT}/wape_by_group.csv")
display(hz.round(4)); display(grp_tab.round(4))

## 11. Lớp quyết định tồn kho

Theo báo cáo mục 7.3.
- Mức tồn kho S = D̂ + z(α)·σₑ, với α ∈ {0.80, 0.90, 0.95}.
- Phương án phi tham số dùng phân vị α của sai số tương đối.
- σₑ lấy từ sai số ngoài mẫu trên tuần validation của mô hình huấn luyện 83 ngày. Sai số đo so với nhu cầu khôi phục D̃. Trên ngày còn hàng đủ, D̃ bằng doanh số.
- Mỗi chuỗi chỉ có 1 mốc validation, tức 7 sai số. Vì vậy σₑ được gộp qua các tầm h và co về nhóm (nhóm quản lý × ngũ phân vị doanh số).

Mô phỏng chính dùng các ngày kiểm tra không có giờ hết hàng 6–22, với D = Y. Phân tích độ nhạy dùng mọi ngày kiểm tra, với D = D̃. Hàng tồn cuối ngày được tính là hao hụt.

In [ ]:
from statistics import NormalDist
with stage("inventory_layer"):
    m_i = np.maximum(DREC[:, :TR_END].mean(1), 0.01)
    q5 = pd.qcut(pd.Series(m_i).rank(method="first"), 5, labels=False).values
    grp_inv = STATIC["management_group_id"] * 10 + q5
    def sigma_and_quant(name, alphas):
        e = DREC[:, VAL] - PRED_VAL[name]                       # N × 7 (dương = thiếu)
        s_i = e.std(1, ddof=1); r = e / m_i[:, None]
        df = pd.DataFrame({"g": np.repeat(grp_inv, 7), "r": r.ravel()})
        cv_g = df.groupby("g")["r"].std().reindex(grp_inv).fillna(np.nanstd(r)).values
        sig = np.sqrt((7 * s_i ** 2 + SIGMA_SHRINK_K * (cv_g * m_i) ** 2) / (7 + SIGMA_SHRINK_K))
        qs = {a: df.groupby("g")["r"].quantile(a).reindex(grp_inv).fillna(np.nanquantile(r, a)).values * m_i for a in alphas}
        return sig, qs
    def inv_metrics(S, D, a):
        short = np.maximum(D - S, 0); over = np.maximum(S - D, 0)
        return {"stockout_rate": float((D > S + 1e-9).mean()), "cycle_service_level": float(1 - (D > S + 1e-9).mean()),
                "fill_rate": float(np.minimum(S, D).sum() / D.sum()), "waste_rate": float(over.sum() / D.sum()),
                "cost": float(a * short.sum() + (1 - a) * over.sum())}
    rows = []
    sims = {"main: test days fully in stock 6-22, D=Y": (FULL16_TEST, Y[:, TEST]),
            "sensitivity: all test days, D=recovered": (np.ones_like(FULL16_TEST, bool), DREC[:, TEST])}
    for name in PRED_TEST:
        sig, qs = sigma_and_quant(name, SERVICE_LEVELS)
        for a in SERVICE_LEVELS:
            z = NormalDist().inv_cdf(a)
            pols = {"normal z*sigma": PRED_TEST[name] + z * sig[:, None],
                    "empirical quantile": PRED_TEST[name] + qs[a][:, None]}
            for pol, S in pols.items():
                S = np.maximum(S, 0)
                for sim, (mask, D) in sims.items():
                    rows.append({"sim": sim, "policy": pol, "alpha": a, "exp": name, **inv_metrics(S[mask], D[mask], a)})
    INV = pd.DataFrame(rows)
    base = INV[INV.exp == "E1"].set_index(["sim", "policy", "alpha"])["cost"]
    INV["rel_newsvendor_cost_vs_E1"] = INV.apply(lambda r: r.cost / base.loc[(r.sim, r.policy, r.alpha)], axis=1)
    INV.to_csv(f"{OUT}/inventory_metrics.csv", index=False)
main = INV[(INV.sim.str.startswith("main")) & (INV.policy == "normal z*sigma")].drop(columns=["sim", "policy", "cost"])
display(main.set_index(["alpha", "exp"]).round(4))

In [ ]:
display(INV[(INV.sim.str.startswith("main")) & (INV.policy == "empirical quantile")]
        .drop(columns=["sim", "policy", "cost"]).set_index(["alpha", "exp"]).round(4))
display(INV[INV.sim.str.startswith("sensitivity") & (INV.policy == "normal z*sigma")]
        .drop(columns=["sim", "policy", "cost"]).set_index(["alpha", "exp"]).round(4))

## 12. Mức độ quan trọng của đặc trưng

Dùng gain và SHAP cho LightGBM E3. Dùng permutation importance cho LSTM E5.

In [ ]:
m3 = LGB_INFO["E3"]["model"]; X3 = LGB_INFO["E3"]["X_test"]
imp = pd.DataFrame({"feature": m3.feature_name(), "gain": m3.feature_importance("gain")})
imp["gain_share"] = imp.gain / imp.gain.sum()
if RUN_SHAP:
    Xs = X3.sample(n=min(20000, len(X3)), random_state=SEED)
    contrib = m3.predict(Xs, pred_contrib=True)[:, :-1]
    imp["mean_abs_shap"] = np.abs(contrib).mean(0)
imp = imp.sort_values("gain", ascending=False); imp.to_csv(f"{OUT}/feature_importance_E3.csv", index=False)
display(imp.head(15).round(4))

if RUN_LSTM:
    info = LSTM_INFO["E5"]; base_pred = predict(info["model"], info["P"], FUT, VA_END - 1) * info["scale"]
    base_w = forecast_metrics(np.maximum(base_pred, 0))["WAPE"]
    pgroups = {"demand": [0], "stockout_cnt": [1], "promo (discount, activity)": [2, 3], "holiday_flag": [4],
               "dow": list(range(5, 12)), "weather": list(range(12, 16))}
    rng = np.random.default_rng(SEED); perm_rows = []
    for g, cols in pgroups.items():
        P = info["P"].copy(); F = FUT.copy(); perm = rng.permutation(N)
        P[:, :, cols] = P[perm][:, :, cols]
        fcols = {"promo (discount, activity)": [0, 1], "holiday_flag": [2], "dow": list(range(3, 10)), "weather": list(range(10, 14))}.get(g, [])
        if fcols: F[:, :, fcols] = F[perm][:, :, fcols]
        pw = forecast_metrics(np.maximum(predict(info["model"], P, F, VA_END - 1) * info["scale"], 0))["WAPE"]
        perm_rows.append({"feature_group": g, "WAPE_permuted": pw, "delta_WAPE": pw - base_w})
    PERM = pd.DataFrame(perm_rows).sort_values("delta_WAPE", ascending=False); PERM.to_csv(f"{OUT}/permutation_importance_E5.csv", index=False)
    display(PERM.round(4))

## 13. Biểu đồ

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
RES["WAPE"].plot.bar(ax=ax[0, 0], color="#2874A6"); ax[0, 0].set_title(f"WAPE trên giờ còn hàng (tập con N={N})"); ax[0, 0].set_ylabel("WAPE")
ax2 = RES["WPE"].plot.bar(ax=ax[0, 1], color="#AF601A"); ax[0, 1].axhline(0, color="k", lw=0.8); ax[0, 1].set_title("WPE (âm = dự báo thấp)")
hz.plot(ax=ax[1, 0], marker="o"); ax[1, 0].set_title("WAPE theo tầm dự báo h"); ax[1, 0].set_xlabel("h (ngày)")
mm = INV[(INV.sim.str.startswith("main")) & (INV.policy == "normal z*sigma")]
for name, g in mm.groupby("exp"):
    ax[1, 1].plot(g.waste_rate, g.fill_rate, marker="o", label=name)
ax[1, 1].set_xlabel("tỷ lệ hao hụt"); ax[1, 1].set_ylabel("fill rate"); ax[1, 1].set_title("Đánh đổi fill rate – hao hụt (α = 0.80/0.90/0.95)"); ax[1, 1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/results_overview.png", dpi=120); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(range(24), PROF[:, 0].mean(0), label="ngày thường"); ax[0].plot(range(24), PROF[:, 1].mean(0), label="cuối tuần")
ax[0].plot(range(24), ST[:, :VA_END].mean((0, 1)), "--", label="tỷ lệ hết hàng"); ax[0].set_xlabel("giờ"); ax[0].legend(); ax[0].set_title("Hồ sơ giờ p(h) trung bình và tỷ lệ hết hàng")
_col = "mean_abs_shap" if "mean_abs_shap" in imp.columns else "gain_share"
top = imp.sort_values(_col, ascending=False).head(12).iloc[::-1]; ax[1].barh(top.feature, top[_col]); ax[1].set_title(f"E3 LightGBM: {_col} (top 12)")
plt.tight_layout(); plt.savefig(f"{OUT}/profile_and_importance.png", dpi=120); plt.show()

## 14. Thời gian, RAM và lưu kết quả

In [ ]:
summary = {"n_series": N, "data_source": SRC, "device": DEV if RUN_LSTM else None, "n_cpu_threads": N_CPU,
           "config": {k: globals()[k] for k in ["SUBSET_N_SERIES", "LSTM_EPOCHS", "LSTM_BATCH_SIZE", "LSTM_HIDDEN", "LSTM_LAYERS",
                                                 "LSTM_SEEDS", "LGB_OBJECTIVE", "LGB_ORIGIN_STRIDE", "REFIT_FULL", "MASK_K", "PPLUS_MIN", "SEED"]},
           "timings_sec": TIMINGS, "ram_gb": RAM, "peak_rss_gb": peak_rss_gb(),
           "lgb": {k: {kk: vv for kk, vv in v.items() if kk in ("best_iter", "n_train_rows", "n_features")} for k, v in LGB_INFO.items()},
           "lstm_per_seed": {k: v["per_seed"].to_dict("records") for k, v in (LSTM_INFO.items() if RUN_LSTM else [])},
           "recovery_stats": rec_stats}
json.dump(summary, open(f"{OUT}/run_summary.json", "w"), indent=1, default=float)
display(pd.DataFrame({"sec": TIMINGS}).join(pd.DataFrame(RAM).T).round(2))
print("peak RSS (GB):", round(peak_rss_gb(), 2), "| outputs ->", OUT)